In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
from datetime import datetime
import matplotlib.pyplot as plt

from sklearn.metrics import confusion_matrix




In [2]:
df=pd.read_csv('data_for_predictions.csv')
df.head()

,Unnamed: 0,id,cons_12m,cons_gas_12m,cons_last_month,forecast_cons_12m,forecast_discount_energy,forecast_meter_rent_12m,forecast_price_energy_off_peak,forecast_price_energy_peak,...,months_modif_prod,months_renewal,channel_MISSING,channel_ewpakwlliwisiwduibdlfmalxowmwpci,channel_foosdfpfkusacimwkcsosbicdxkicaua,channel_lmkebamcaaclubfxadlmueccxoimlema,channel_usilxuppasemubllopkaafesmlibmsdf,origin_up_kamkkxfxxuwbdslkwifmmcsiusiuosws,origin_up_ldkssxwpmemidmecebumciepifcamkci,origin_up_lxidpiddsbxsbosboudacockeimpuepw
0,0,24011ae4ebbe3035111d65fa7c15bc57,0.000000,4.739944,0.000000,0.000000,0.0,0.444045,0.114481,0.098142,...,2,6,0,0,1,0,0,0,0,1
1,1,d29c2c54acc38ff3c0614d0a653813dd,3.668479,0.000000,0.000000,2.280920,0.0,1.237292,0.145711,0.000000,...,76,4,1,0,0,0,0,1,0,0
2,2,764c75f661154dac3a6c254cd082ea7d,2.736397,0.000000,0.000000,1.689841,0.0,1.599009,0.165794,0.087899,...,68,8,0,0,1,0,0,1,0,0
3,3,bba03439a292a1e166f80264c16191cb,3.200029,0.000000,0.000000,2.382089,0.0,1.318689,0.146694,0.000000,...,69,9,0,0,0,1,0,1,0,0
4,4,149d57cf92fc41cf94415803a877cb4b,3.646011,0.000000,2.721811,2.650065,0.0,2.122969,0.116900,0.100015,...,71,9,1,0,0,0,0,1,0,0


In [3]:
df=df.drop(columns=['Unnamed: 0'])

In [4]:

from sklearn import metrics
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier


In [5]:
#SPLITTING TRAIN AND TEST
train_df = df.copy()

# Separate target variable from independent variables
y = df['churn']
X = df.drop(columns=['id', 'churn'])
print(X.shape)
print(y.shape)

(14606, 61)
(14606,)


In [6]:
X_train,X_test,y_train,y_test=train_test_split(X,y, test_size=0.25, random_state=42)
print(X_train.shape)
print(y_train.shape)
print(X_test.shape)
print(y_test.shape)

(10954, 61)
(10954,)
(3652, 61)
(3652,)


In [7]:
#MODELING
model=RandomForestClassifier(n_estimators=200,  class_weight='balanced',random_state=42, )


In [8]:
model.fit(X_train, y_train)

RandomForestClassifier(class_weight='balanced', n_estimators=200,
                       random_state=42)

In [9]:
y_pred=model.predict(X_test)

In [10]:
metrics.confusion_matrix(y_test, y_pred)

array([[3282,    4],
       [ 346,   20]])

In [ ]:
# here we did identify 3167 not churn correctly
# but we did not correctly identify 308 churned customer.

In [55]:
print(metrics.classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.90      1.00      0.95      3286
           1       0.83      0.05      0.10       366

    accuracy                           0.90      3652
   macro avg       0.87      0.53      0.53      3652
weighted avg       0.90      0.90      0.86      3652



In [41]:
feature_importances = pd.DataFrame({
    'features': X_train.columns,
    'importance': model.feature_importances_
}).sort_values(by='importance', ascending=True).reset_index()


In [45]:
feature_importances.tail(10)

,index,features,importance
51,51,months_modif_prod,0.030401
52,16,var_year_price_off_peak_var,0.031248
53,2,cons_last_month,0.036537
54,3,forecast_cons_12m,0.036850
55,14,net_margin,0.037983
56,49,months_activ,0.038093
57,5,forecast_meter_rent_12m,0.042210
58,0,cons_12m,0.050849
59,11,margin_gross_pow_ele,0.058249
60,12,margin_net_pow_ele,0.061214


In [ ]:
'''### Evaluation

We are going to use 3 metrics to evaluate performance:

- Accuracy = the ratio of correctly predicted observations to the total observations
- Precision = the ability of the classifier to not label a negative sample as positive
- Recall = the ability of the classifier to find all the positive samples
'''

In [58]:
df.groupby(['churn'])['margin_net_pow_ele'].describe()

,count,mean,std,min,25%,50%,75%,max
churn,,,,,,,,
0,13187.0,23.926979,19.357893,0.0,13.835,21.48,29.64,374.64
1,1419.0,30.468682,26.306684,0.0,17.130,26.04,34.68,299.64


In [59]:
df.groupby(['churn'])['margin_gross_pow_ele'].describe()

,count,mean,std,min,25%,50%,75%,max
churn,,,,,,,,
0,13187.0,23.929863,19.359021,0.0,13.86,21.48,29.64,374.64
1,1419.0,30.468682,26.306684,0.0,17.13,26.04,34.68,299.64


In [65]:
df.groupby(['churn'])['cons_12m'].describe()

,count,mean,std,min,25%,50%,75%,max
churn,,,,,,,,
0,13187.0,4.228782,0.892816,0.0,3.751741,4.148479,4.610282,6.792889
1,1419.0,4.178929,0.802169,0.0,3.766190,4.162266,4.610201,6.597250
